### multimodel measn test+ images RAG means pipline 

doc--> chenks ---> vector represntaion---> vectore ---> retrvier --> based on qurey (embendings) ---> top k data ---> to llm --> genrate the output

CLIP --->model provided by open ai  trained 400b huggingface vible help to convert to text and image data text vectore create

architecture of this project

pdfdoc                                                                          QUERY
 |                                                                                  |
chenks(text)(embeding)                                                          CLIP(Embeddingd)
   |                    -->this both(store in FAISS)<--------RETRIVER ---->     <---
images(embendings)                                                             |
                                                                            formate(based on top k text images)
                                                                                        |
                                                                                        llm

clip --> contrastive language image pre training
architecture inside is ---> vision transformer + transerformer


PYMUPDF---> WE ARE USING GOOD FOR TEXT IMAGE ETC EXTRACTION

In [ ]:
import numpy as np
import torch
from langchain_core.documents import  Document
from PIL import  Image
from langchain.chat_models import  init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import  HumanMessage
from transformers import CLIprocessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

import os 
import base64
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS



ImportError: DLL load failed while importing _C: The specified module could not be found.

In [ ]:
### clip mode
import os 
from dotenv import load_dotenv
load_dotenv()
## set up env
os.environ["OPENAI_API_KEY"]=os.getenv("OPEN_API_KEY")

### INTILIZE RHE CLIP MODEL 

clip_mode=CLIPModeL.from_pretrained("openai/clip-vit-base-patch32")
clip_processer=CLIprocessor.from_pretrained("openai/clip-vit-base-patch32"  )   # for froate convert
clip_mode.eval()


In [ ]:
###emdingthe images 
def image_emding(image_data):
    "embeding image usng clip"
    if isinstance(image_data, str):
        image=Image.open(image_data).convert("RGB")
    else:  # if base64 formate
        image=image_data
# concvert to embddings
    imput=clip_processer(image=image, return_tensors="pt")
    with torch.no_grad():
        clip_mode.get_image_feature(**input)
        #normlize embbeding to unit vectors
        #bceause the iamge size is diffrent basicaly convert to -1 dimision 
        feature=feature/feature.norm(dim=-1, keepdim=True)
        return feature.squeez().numpy()
    

def texr_emding(image_data):
    "embeding image usng clip"
    clip_processer(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77   #clip max token length

    )
# concvert to embddings
    with torch.no_grad():
        clip_mode.get_text_feature(**input)
        #normlize embbeding to unit vectors
        #bceause the text size is diffrent basicaly convert to -1 dimision 
        feature=feature/feature.norm(dim=-1, keepdim=True)
        return feature.squeez().numpy()

In [ ]:
padf_path="filepath pdf"
doc=fitz.open(padf_path)
## store all doc and embdi
all_doc=[]
all_embdding=[]
image_data_store={}
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [ ]:
for i , page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ## create a temp doc splitting
        temp_doc=Document(page_content=text, metadata={"page":i, "type":"text"})
        text_chunks=splitter.split_documents([temp_doc])
        # embb in each chnuks
        for chunk in text_chunks:
            embbeding=embedd_text(chunk.page_content)
            all_embdding.append(embbeding)
            all_doc.append(chunk)


In [ ]:
## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embdding.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_doc.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

In [2]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embdding)
embeddings_array

NameError: name 'all_embdding' is not defined

In [ ]:
# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

In [ ]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

In [ ]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [ ]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)